# Aetheris Health — ISIC lesion-risk baseline

**Purpose:** train an assistive lesion-risk classifier, not a diagnostic model. This notebook accepts only the ISIC 2024 *Permissive* dataset (CC-BY), validates its schema, makes patient-level splits, calibrates a review threshold, and exports a versioned checkpoint.

**Patient-facing policy:** never display the raw model probability or a cancer diagnosis. A high-risk or uncertain result routes to a clinician.

## Kaggle setup

1. In **Add Input**, attach only `roscoekerby/isic-2024-permissive-training-input`.
2. Set Accelerator to **GPU T4 x2**. This notebook uses both GPUs when Kaggle exposes them.
3. Turn Internet on only if the ImageNet EfficientNet weights are not already cached.
4. Run the dependency cell, then use **Restart Session** before continuing. This is required after replacing Pillow.
5. Never attach, upload, or train on patient-identifiable Aetheris images in Kaggle.

In [ ]:
# Pin Pillow exactly: Kaggle's torchvision build can fail with newer Pillow releases.
# After this cell completes, select Run > Restart Session, then resume at the next cell.
!pip -q install --no-cache-dir --force-reinstall "Pillow==10.4.0"
!pip -q install --no-cache-dir "timm==1.0.15" "scikit-learn==1.5.2" "albumentations==1.4.18"


In [ ]:
from pathlib import Path
import json, os, random, warnings
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 20260907
IMAGE_SIZE = 224
BATCH_SIZE_PER_GPU = 64       # total 128 on Kaggle T4 x2; reduce to 48 only on CUDA out-of-memory
EVAL_BATCH_SIZE_PER_GPU = 128  # total 256 on Kaggle T4 x2
EPOCHS = 4                     # baseline; increase only after verifying throughput and validation trend
NUM_WORKERS = 8                # JPEG decoding is the bottleneck on Kaggle's mounted input
OUTPUT_DIR = Path('/kaggle/working/aetheris_isic_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print({'device': str(DEVICE), 'gpu_count': N_GPUS, 'torch': torch.__version__, 'cpu_count': os.cpu_count()})
assert DEVICE.type == 'cuda', 'Enable a Kaggle GPU accelerator before training.'
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv,noheader


In [ ]:
# Discover files first. This path matches the mounted Kaggle input supplied for Aetheris.
ROOT = Path('/kaggle/input/datasets/roscoekerby/isic-2024-permissive-training-input')
assert ROOT.exists(), f'Attach the exact Kaggle input first: {ROOT}'
csv_files = sorted(ROOT.rglob('*.csv'))
image_files = list(ROOT.rglob('*.jpg')) + list(ROOT.rglob('*.jpeg')) + list(ROOT.rglob('*.png'))
print('CSV files:', [str(p.relative_to(ROOT)) for p in csv_files])
print('Image files:', len(image_files))
assert csv_files and image_files, 'Expected metadata CSV files and image files.'
for p in csv_files:
    preview = pd.read_csv(p, nrows=3)
    print('\n', p.name, 'columns:', preview.columns.tolist())


In [ ]:
# This Kaggle mirror may contain only `isic_id` and `malignant`.
# If official patient metadata is attached, the notebook uses it. Otherwise it creates an IMAGE-LEVEL EXPERIMENTAL split.
# Never use image-level split results as clinical or production performance claims.
ALLOW_IMAGE_LEVEL_EXPERIMENTAL_SPLIT = True

def first_column(frame, names, description, required=True):
    match = next((name for name in names if name in frame.columns), None)
    if match is None and required:
        raise ValueError(f'Missing {description}. Found: {frame.columns.tolist()}')
    return match

candidates = []
for p in csv_files:
    df = pd.read_csv(p)
    target = next((c for c in ['target', 'malignant', 'malignancy', 'is_malignant'] if c in df.columns), None)
    if target is not None:
        candidates.append((p, df, target))
assert len(candidates) == 1, 'Expected exactly one CSV containing the binary malignancy target.'
LABEL_PATH, meta, TARGET_COL = candidates[0]
IMAGE_ID_COL = first_column(meta, ['isic_id', 'image_id', 'image_name'], 'image identifier')
PATIENT_ID_COL = first_column(meta, ['patient_id', 'patient', 'patient_identifier'], 'patient identifier', required=False)
meta[TARGET_COL] = pd.to_numeric(meta[TARGET_COL], errors='raise').astype('int8')
assert set(meta[TARGET_COL].unique()).issubset({0, 1}), 'The target must be binary 0/1.'

if PATIENT_ID_COL is None:
    assert ALLOW_IMAGE_LEVEL_EXPERIMENTAL_SPLIT, 'Attach official ISIC metadata with patient_id or explicitly allow an image-level experimental split.'
    SPLIT_GROUP_COL, SPLIT_MODE = IMAGE_ID_COL, 'image_level_experimental__patient_metadata_missing'
    print('WARNING: patient_id is absent. This run is experimental and cannot support clinical/production performance claims.')
else:
    SPLIT_GROUP_COL, SPLIT_MODE = PATIENT_ID_COL, 'patient_level'
print({'metadata': LABEL_PATH.name, 'rows': len(meta), 'positive_rate': float(meta[TARGET_COL].mean()), 'image_id': IMAGE_ID_COL, 'patient_id': PATIENT_ID_COL, 'split_mode': SPLIT_MODE, 'target': TARGET_COL})


In [ ]:
# Map every metadata row to exactly one local image. Missing images fail closed.
paths_by_stem = {}
for p in image_files:
    paths_by_stem.setdefault(p.stem, p)
meta['image_path'] = meta[IMAGE_ID_COL].astype(str).map(lambda x: str(paths_by_stem.get(x, '')))
missing = (meta.image_path == '').sum()
assert missing == 0, f'{missing} metadata rows have no matching image; inspect this Kaggle mirror before continuing.'
assert meta[IMAGE_ID_COL].is_unique, 'Expected one metadata row per image.'

# 70/15/15 split. It is patient-level only when patient metadata is mounted.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, holdout_idx = next(splitter.split(meta, groups=meta[SPLIT_GROUP_COL]))
train_df, holdout_df = meta.iloc[train_idx].copy(), meta.iloc[holdout_idx].copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)
val_idx, test_idx = next(splitter.split(holdout_df, groups=holdout_df[SPLIT_GROUP_COL]))
val_df, test_df = holdout_df.iloc[val_idx].copy(), holdout_df.iloc[test_idx].copy()
for name, part in [('train',train_df),('validation',val_df),('test',test_df)]:
    print(name, len(part), f'{SPLIT_GROUP_COL}s=', part[SPLIT_GROUP_COL].nunique(), 'positive_rate=', round(part[TARGET_COL].mean(),5))
assert set(train_df[SPLIT_GROUP_COL]).isdisjoint(val_df[SPLIT_GROUP_COL])
assert set(train_df[SPLIT_GROUP_COL]).isdisjoint(test_df[SPLIT_GROUP_COL])
assert set(val_df[SPLIT_GROUP_COL]).isdisjoint(test_df[SPLIT_GROUP_COL])


In [ ]:
# ISIC lesion crops are already centred. Use lightweight training transforms so CPU JPEG processing does not starve the GPUs.
train_tfms = A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(0.08, 0.08, p=0.20), A.Normalize(), ToTensorV2()])
eval_tfms = A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.Normalize(), ToTensorV2()])

class LesionDataset(Dataset):
    # Store plain arrays rather than accessing a Pandas row for every image.
    def __init__(self, frame, transforms):
        self.paths = frame['image_path'].tolist(); self.targets = frame[TARGET_COL].astype('float32').to_numpy(); self.ids = frame[IMAGE_ID_COL].astype(str).tolist(); self.transforms = transforms
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as source:
            image = np.asarray(source.convert('RGB'))
        return self.transforms(image=image)['image'], torch.tensor(self.targets[index]), self.ids[index]

def loader(frame, train=False):
    per_gpu = BATCH_SIZE_PER_GPU if train else EVAL_BATCH_SIZE_PER_GPU
    return DataLoader(LesionDataset(frame, train_tfms if train else eval_tfms), batch_size=per_gpu * max(1,N_GPUS), shuffle=train, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
train_loader, val_loader, test_loader = loader(train_df, True), loader(val_df), loader(test_df)
print({'train_batches':len(train_loader), 'validation_batches':len(val_loader), 'test_batches':len(test_loader), 'train_batch_size':BATCH_SIZE_PER_GPU * max(1,N_GPUS), 'inference_batch_size':EVAL_BATCH_SIZE_PER_GPU * max(1,N_GPUS), 'num_workers':NUM_WORKERS})


In [ ]:
# EfficientNet-B0 is a baseline. Its output is a risk score, not a diagnosis.
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=1).to(DEVICE)
if N_GPUS > 1: model = nn.DataParallel(model)
positive_weight = torch.tensor([(len(train_df)-train_df[TARGET_COL].sum()) / train_df[TARGET_COL].sum()], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

@torch.inference_mode()
def predict(data_loader, description='Inference'):
    model.eval(); probabilities, labels, identifiers = [], [], []
    for images, targets, ids in tqdm(data_loader, desc=description, leave=False):
        with torch.amp.autocast('cuda'):
            logits = model(images.to(DEVICE, non_blocking=True)).flatten()
        probabilities.extend(torch.sigmoid(logits).cpu().numpy()); labels.extend(targets.numpy()); identifiers.extend(ids)
    return np.asarray(probabilities), np.asarray(labels), identifiers

best_auc, history = -np.inf, []
for epoch in range(1, EPOCHS + 1):
    model.train(); running_loss = 0.0
    for images, targets, _ in tqdm(train_loader, desc=f'Training epoch {epoch}/{EPOCHS}', leave=False):
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(images.to(DEVICE, non_blocking=True)).flatten()
            loss = criterion(logits, targets.to(DEVICE, non_blocking=True))
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); running_loss += loss.item() * len(images)
    scheduler.step()
    val_p, val_y, _ = predict(val_loader, f'Validation epoch {epoch}/{EPOCHS}'); val_auc = roc_auc_score(val_y, val_p); val_ap = average_precision_score(val_y, val_p)
    record = {'epoch':epoch,'train_loss':running_loss/len(train_df),'val_auc':val_auc,'val_ap':val_ap}; history.append(record); print(record)
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({'model_state': (model.module if isinstance(model,nn.DataParallel) else model).state_dict(), 'epoch':epoch, 'val_auc':val_auc,'split_mode':SPLIT_MODE, 'config':{'model':'efficientnet_b0','image_size':IMAGE_SIZE}}, OUTPUT_DIR/'best_uncalibrated.pt')
pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv', index=False)


In [ ]:
# This runs two full, visible inference passes. Calibration itself is fast.
base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(torch.load(OUTPUT_DIR/'best_uncalibrated.pt', map_location=DEVICE)['model_state'])
print('Running validation inference for calibration…')
val_p, val_y, _ = predict(val_loader, 'Calibration: validation')
calibrator = LogisticRegression(C=1.0).fit(val_p.reshape(-1,1), val_y)
val_cal = calibrator.predict_proba(val_p.reshape(-1,1))[:,1]
thresholds = np.linspace(0.01, 0.99, 99)
# Select the lowest threshold attaining >=95% validation sensitivity; final threshold needs clinician approval.
sensitivity = [((val_cal >= t) & (val_y == 1)).sum() / max(1, (val_y == 1).sum()) for t in thresholds]
review_threshold = float(thresholds[next((i for i,s in enumerate(sensitivity) if s >= 0.95), len(thresholds)-1)])
print({'validation_auc':roc_auc_score(val_y,val_cal), 'review_threshold':review_threshold, 'validation_sensitivity':max(sensitivity), 'split_mode':SPLIT_MODE})

print('Running held-out test inference…')
test_p, test_y, test_ids = predict(test_loader, 'Final test inference')
test_cal = calibrator.predict_proba(test_p.reshape(-1,1))[:,1]
pred = (test_cal >= review_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(test_y, pred, labels=[0,1]).ravel()
metrics = {'test_auc':float(roc_auc_score(test_y,test_cal)), 'test_average_precision':float(average_precision_score(test_y,test_cal)), 'sensitivity':float(tp/max(1,tp+fn)), 'specificity':float(tn/max(1,tn+fp)), 'false_negatives':int(fn), 'threshold':review_threshold, 'split_mode':SPLIT_MODE}
print(metrics)

torch.save({'model_state':base_model.state_dict(), 'calibrator':calibrator, 'review_threshold':review_threshold, 'metrics':metrics, 'dataset':'ISIC 2024 SLICE-3D Permissive', 'license':'CC-BY 4.0; attribution required', 'split_mode':SPLIT_MODE, 'intended_use':'assistive lesion-risk routing only'}, OUTPUT_DIR/'aetheris_isic_lesion_risk_v1.pt')
pd.DataFrame({'image_id':test_ids,'target':test_y,'calibrated_risk':test_cal,'priority_review':pred, 'split_mode':SPLIT_MODE}).to_csv(OUTPUT_DIR/'heldout_test_predictions.csv',index=False)
print('Exported:', list(OUTPUT_DIR.iterdir()))


## Release gate

Do **not** integrate this checkpoint into the patient app until a dermatologist reviews the held-out errors, approves the threshold, and documents its limits. This model is trained on lesion crops; it is not validated for all rashes or arbitrary phone photographs.